# 03. Exploratory Data Analysis (EDA)

## Objectives:
1. Compare transaction amounts between legitimate and fraudulent transactions.
2. Explore cyclical transaction activity across hours of the day.
3. Analyze discriminative power of key PCA features (V14, V17, V12, V4).
4. Highlight key statistical insights for business stakeholders.


In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.config import CLEANED_DATA_PATH

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
df = pd.read_csv(CLEANED_DATA_PATH)
print(f'Loaded {len(df):,} cleaned records.')


### Transaction Amount Distribution (Log-Scale Contrast)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df[df['Class'] == 0]['Amount'], bins=50, kde=True, ax=axes[0], color='#10B981', log_scale=True)
axes[0].set_title('Legitimate Transactions (Amount)', fontweight='bold')
axes[0].set_xlabel('Amount ($)')

sns.histplot(df[df['Class'] == 1]['Amount'], bins=50, kde=True, ax=axes[1], color='#EF4444', log_scale=True)
axes[1].set_title('Fraudulent Transactions (Amount)', fontweight='bold')
axes[1].set_xlabel('Amount ($)')
plt.tight_layout()
plt.show()


### Temporal Fraud Distribution by Hour of Day


In [ ]:
df['hour'] = ((df['Time'] // 3600) % 24).astype(int)
hourly = df.groupby('hour')['Class'].agg(['count', 'sum']).reset_index()
hourly.columns = ['Hour', 'Total', 'Fraud']
hourly['Fraud_Rate_Pct'] = (hourly['Fraud'] / hourly['Total']) * 100

plt.figure(figsize=(12, 4.5))
sns.barplot(data=hourly, x='Hour', y='Fraud_Rate_Pct', color='#2563EB')
plt.title('Fraud Rate (%) Across 24-Hour Diurnal Cycle', fontweight='bold')
plt.ylabel('Fraud Rate (%)')
plt.xlabel('Hour of Day (0 - 23)')
plt.tight_layout()
plt.show()


### Top Discriminative PCA Components (V14, V17, V12, V4)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
feats = ['V14', 'V17', 'V12', 'V4']
for idx, f in enumerate(feats):
    ax = axes[idx // 2, idx % 2]
    sns.kdeplot(df[df['Class'] == 0][f], label='Legit (0)', color='#10B981', fill=True, alpha=0.3, ax=ax)
    sns.kdeplot(df[df['Class'] == 1][f], label='Fraud (1)', color='#EF4444', fill=True, alpha=0.3, ax=ax)
    ax.set_title(f'Distribution of {f}', fontweight='bold')
    ax.legend()
plt.tight_layout()
plt.show()
